Aktil Bansal Q-1

In [0]:
%sql
create table b_sql.b_practice.icc_world_cup
(
Team_1 Varchar(20),
Team_2 Varchar(20),
Winner Varchar(20)
);
INSERT INTO b_sql.b_practice.icc_world_cup values('India','SL','India');
INSERT INTO b_sql.b_practice.icc_world_cup values('SL','Aus','Aus');
INSERT INTO b_sql.b_practice.icc_world_cup values('SA','Eng','Eng');
INSERT INTO b_sql.b_practice.icc_world_cup values('Eng','NZ','NZ');
INSERT INTO b_sql.b_practice.icc_world_cup values('Aus','India','India');

select * from b_sql.b_practice.icc_world_cup;

In [0]:
%sql
with cte as(select team_1 as team, case when team_1=winner then 1 else 0 end as win_flag from b_sql.b_practice.icc_world_cup
union all
select team_2, case when team_2=winner then 1 else 0 end as win_flag from b_sql.b_practice.icc_world_cup
order by team)
select team,count(*) as total_match,sum(win_flag) as win, count(*)-sum(win_flag) as loss  from cte group by team order by win desc

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
df_match=spark.read.table("b_sql.b_practice.icc_world_cup")
df_match.display()

In [0]:
# df_team1=df_match.withColumn("win_flag",when(col("team_1")==col("Winner"),1).otherwise(0)).select(col("team_1").alias("team"),"win_flag")
# df_team2=df_match.withColumn("win_flag",when(col("team_2")==col("Winner"),1).otherwise(0)).select(col("team_2").alias("team"),"win_flag")

df_team1 = df_match.select(
    col("team_1").alias("team"),
    expr("CASE WHEN team_1 = Winner THEN 1 ELSE 0 END as win_flag")
)
df_team2 = df_match.select(
    col("team_2").alias("team"),
    expr("CASE WHEN team_2 = Winner THEN 1 ELSE 0 END").alias("win_flag")
)

# df_team1 = df_match.select(
#     col("team_1").alias("team"),
#     when(col("team_1") == col("Winner"), 1).otherwise(0).alias("win_flag")
# )
# df_team2 = df_match.select(
#     col("team_2").alias("team"),
#     when(col("team_2") == col("Winner"), 1).otherwise(0).alias("win_flag")
# )
df_team=df_team1.unionAll(df_team2)
df_final=df_team.groupBy("team").agg(count("*").alias("total_match"),sum("win_flag").alias("win"),(count("*")-sum("win_flag")).alias("loss")).orderBy(col("win").desc())
df_final.display()